# unbind-tuple-unpack — worked example 3: Two-level unbind of a ray batch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `unbind-tuple-unpack`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Nested `unbind` destructures multi-axis tensors. A `(B, 2, 3)` ray batch unbinds along `dim=1` into origin and direction, each `(B, 3)`; unbinding those along `dim=-1` yields six named `(B,)` component tensors. This is the ARENA ray-tracing idiom that keeps the analytic solver readable.

## Worked solution

We fully destructure a ray batch into six scalar-per-ray components.

1. First level: `rays.unbind(dim=1)` splits the size-2 axis into `(origin, direction)`, each `(B, 3)`.
2. Second level: `origin.unbind(dim=-1)` and `direction.unbind(dim=-1)` split each into three `(B,)` components.
3. We tuple-unpack into `ox, oy, oz` and `dx, dy, dz`, giving named handles for downstream per-axis math.

We print the six components for a tiny batch to confirm every axis was peeled correctly.

In [ ]:
import torch as t

rays = t.tensor([
    [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
    [[1.0, 1.0, 1.0], [0.0, 1.0, 0.0]],
])

def destructure(rays: t.Tensor) -> dict:
    origin, direction = rays.unbind(dim=1)
    ox, oy, oz = origin.unbind(dim=-1)
    dx, dy, dz = direction.unbind(dim=-1)
    return {'ox': ox, 'oy': oy, 'oz': oz, 'dx': dx, 'dy': dy, 'dz': dz}

out = destructure(rays)
print('ox:', out['ox'].tolist())
print('dy:', out['dy'].tolist())